In [ ]:
!git clone https://github.com/Ish2905/Comemo-Dataset.git

In [ ]:
%cd /content/Comemo-Dataset

In [ ]:
!git pull

In [ ]:
# ==== DO NOT MODIFY THIS CELL ====
from google.colab import drive
drive.mount('/content/drive')

import duckdb
import os

DB_PATH = "/content/drive/MyDrive/Capstone/comemo.db"

# HARD FAIL if Drive is not mounted
assert os.path.exists("/content/drive/MyDrive"), "Drive not mounted!"

# HARD FAIL if DB file missing (after first creation)
if not os.path.exists(DB_PATH):
    print("⚠️ comemo.db not found yet (first run only)")
else:
    print("✅ Using existing database:", DB_PATH)

con = duckdb.connect(DB_PATH)

# Sanity check
print(con.execute("SHOW TABLES").fetchdf())
# =================================


In [ ]:
con.execute("""
SELECT
  'reviews_raw' AS table,
  COUNT(*) AS rows
FROM reviews_raw
UNION ALL
SELECT
  'metadata_raw',
  COUNT(*)
FROM metadata_raw
""").fetchdf()


In [ ]:
con.execute("DESC metadata_raw").fetchdf()

In [ ]:
con.execute("DESC reviews_raw").fetchdf()

In [ ]:
con.execute("""
COPY reviews_raw
TO '/content/drive/MyDrive/Capstone/reviews_raw.parquet'
(FORMAT PARQUET);

COPY metadata_raw
TO '/content/drive/MyDrive/Capstone/metadata_raw.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("PRAGMA threads=4;")
con.execute("PRAGMA enable_progress_bar=true;")



In [ ]:
con.execute("""
CREATE OR REPLACE TABLE reviews_monthly AS
WITH base AS (
    SELECT
        parent_asin,
        DATE_TRUNC('month', to_timestamp(timestamp)) AS month_start,
        rating
    FROM reviews_raw
),

first_review AS (
    SELECT
        parent_asin,
        MIN(month_start) AS first_month
    FROM base
    GROUP BY parent_asin
)

SELECT
    b.parent_asin,

    EXTRACT(year FROM b.month_start) AS year,
    EXTRACT(month FROM b.month_start) AS month,
    STRFTIME(b.month_start, '%Y-%m') AS year_month,

    COUNT(*) AS monthly_review_count,
    AVG(b.rating) AS monthly_rating_avg,

    DATE_DIFF(
        'month',
        f.first_month,
        b.month_start
    ) AS product_age_months

FROM base b
JOIN first_review f
    ON b.parent_asin = f.parent_asin

GROUP BY
    b.parent_asin,
    year,
    month,
    year_month,
    b.month_start,
    f.first_month;
""")

In [ ]:
con.execute("SHOW TABLES").fetchdf()


In [ ]:
con.execute("DESC reviews_monthly").fetchdf()